In [ ]:
# 실습에 필요한 패키지를 설치합니다.
!pip install -r ../requirements.txt

# 04 LangGraph

LangGraph는 LLM 애플리케이션의 흐름을 `상태(state)` 중심으로 설계하고 실행하는 그래프 기반 orchestration 도구다.
간단한 체인처럼 한 방향으로만 흘러가는 작업이 아니라, 상태를 읽고 다음 단계를 결정해야 하는 workflow를 만들 때 특히 강하다.

예를 들어 어떤 질문은 바로 답변하고, 어떤 질문은 검색으로 보내고, 어떤 경우에는 다시 시도하거나 사람 검토로 넘겨야 할 수 있다.
LangGraph는 이런 흐름을 `node`, `edge`, `START`, `END`, 그리고 공유 `state`로 표현하게 해준다.

## 이번 노트에서 볼 것

- LangGraph란 무엇인가
- `StateGraph`와 `state schema`
- `node`, `edge`, `START`, `END`
- 각 node가 state를 어떻게 읽고 갱신하는가
- 왜 체인보다 graph가 필요한가

## 1. LangGraph란 무엇인가

LangGraph는 여러 처리 단계를 `노드(node)`와 `엣지(edge)`로 연결하고, 그 사이를 흐르는 `상태(state)`를 기준으로 다음 동작을 결정하는 프레임워크다.

조금 더 직관적으로 말하면,

- node: 실제 일을 하는 단계
- edge: 다음에 어디로 이동할지 연결하는 선
- state: 각 단계가 함께 읽고 갱신하는 공유 정보
- graph: 이 전체 흐름을 표현한 실행 지도

중요한 점은 LangGraph가 단순히 함수를 순서대로 호출하는 도구가 아니라, 현재 상태를 보고 다음 단계를 결정할 수 있는 구조라는 점이다.
즉 `A -> B -> C`처럼 한 줄로 끝나는 흐름뿐 아니라, 조건 분기나 반복이 있는 workflow도 자연스럽게 표현할 수 있다.

## 2. StateGraph와 state schema

`StateGraph`는 여러 node가 함께 읽고 갱신할 공유 상태를 기준으로 workflow를 구성하는 그래프다.

여기서 가장 먼저 해야 할 일은 `state schema`를 정의하는 것이다.
state schema는 "이 workflow에서 어떤 정보를 상태로 관리할 것인가"를 명시하는 설계도다.

이번 예제에서는 아래 상태를 사용한다.

- `topic`: 지금 공부할 주제
- `notes`: 각 node가 누적해서 남기는 메모
- `understanding`: 현재 이해 단계
- `need_example`: 예시 단계가 필요한지 여부

`notes`는 여러 node가 차곡차곡 내용을 추가해야 하므로 reducer를 함께 지정한다.

In [ ]:
from operator import add
from pprint import pprint
from typing import Annotated, TypedDict


class StudyState(TypedDict):
    topic: str
    notes: Annotated[list[str], add]
    understanding: str
    need_example: bool


StudyState.__annotations__

위 코드에서 `Annotated[list[str], add]`는 `notes` 필드에 여러 node가 리스트를 더해도 안전하게 합쳐지도록 만드는 reducer 설정이다.

즉,
- 어떤 node가 `{"notes": ["첫 메모"]}`를 반환하고
- 다음 node가 `{"notes": ["둘째 메모"]}`를 반환하면
- 최종 state의 `notes`는 두 항목이 합쳐진 리스트가 된다

## 3. node, edge, START, END

이제 실제 node를 만들어 보자.

이번 예제의 흐름은 아래와 같다.

1. `START`에서 시작
2. `explain_graph`가 graph 개념을 메모에 추가
3. `explain_state`가 state 개념을 메모에 추가
4. 현재 state를 보고 예시가 필요하면 `show_example`로, 아니면 `finish`로 이동
5. `finish`에서 정리하고 `END`로 종료

여기서 `START`와 `END`는 특별한 예약 노드로, 실행의 시작점과 종료점을 나타낸다.

In [ ]:
from langgraph.graph import END, START, StateGraph


def explain_graph(state: StudyState):
    topic = state["topic"]
    return {
        "notes": [f"Graph는 {topic} 학습 단계를 node와 edge로 표현한 구조다."],
        "understanding": "graph basics",
    }


def explain_state(state: StudyState):
    need_example = "LangGraph" in state["topic"]
    return {
        "notes": ["State는 각 단계가 함께 읽고 갱신하는 공유 작업 메모리다."],
        "need_example": need_example,
        "understanding": "state basics",
    }


def route_after_state(state: StudyState):
    return "show_example" if state["need_example"] else "finish"


def show_example(state: StudyState):
    return {
        "notes": ["예: START -> explain_graph -> explain_state -> show_example -> finish -> END"],
        "understanding": "example added",
    }


def finish(state: StudyState):
    return {
        "notes": ["END에 도달하면 이번 실행이 종료된다."],
        "understanding": "week1 complete",
    }

In [ ]:
graph = StateGraph(StudyState)

graph.add_node("explain_graph", explain_graph)
graph.add_node("explain_state", explain_state)
graph.add_node("show_example", show_example)
graph.add_node("finish", finish)

graph.add_edge(START, "explain_graph")
graph.add_edge("explain_graph", "explain_state")
graph.add_conditional_edges(
    "explain_state",
    route_after_state,
    {
        "show_example": "show_example",
        "finish": "finish",
    },
)
graph.add_edge("show_example", "finish")
graph.add_edge("finish", END)

app = graph.compile()

In [ ]:
print(app.get_graph().draw_mermaid())

## 4. 각 node는 state를 어떻게 읽고 갱신하는가

LangGraph에서 node는 현재 state를 입력으로 받고, 전체 state를 통째로 다시 만드는 대신 `바뀐 부분만` 반환하면 된다.

예를 들어 `explain_state` node는 아래처럼 동작한다.

- 입력으로 현재 `state`를 받는다
- `state["topic"]`을 읽어서 예시가 필요한지 판단한다
- `notes`, `need_example`, `understanding`의 새 값을 반환한다
- 반환된 값은 기존 state와 합쳐져 다음 node의 입력이 된다

즉 node의 핵심 책임은 "현재 상태를 읽고, 다음 상태를 위한 업데이트를 반환하는 것"이다.

In [ ]:
initial_state = {
    "topic": "LangGraph 핵심 구조",
    "notes": [],
    "understanding": "start",
    "need_example": False,
}

for event in app.stream(initial_state, stream_mode="updates"):
    pprint(event)

In [ ]:
result = app.invoke(initial_state)
pprint(result)

## 5. 왜 체인보다 graph가 필요한가

체인은 보통 고정된 순서의 단계를 표현할 때 잘 맞는다.
예를 들어 `질문 정리 -> 검색 -> 답변 생성`처럼 경로가 하나라면 체인만으로도 충분할 수 있다.

하지만 실제 workflow는 자주 아래 문제를 만난다.

- 어떤 상태에서는 다른 경로로 보내야 한다
- 결과가 부족하면 다시 이전 단계로 돌아가야 한다
- 특정 조건에서만 사람 검토를 받아야 한다
- 실패 시 fallback 경로가 필요하다

이런 경우에는 상태를 보고 다음 노드를 결정하는 graph가 더 자연스럽다.

간단히 비교하면,

- chain은 "정해진 순서의 실행"에 강하다
- graph는 "상태에 따라 바뀌는 실행"에 강하다

상태 기반 orchestration을 공부할 때 LangGraph가 중요한 이유도 여기에 있다.
LLM이 들어가더라도 본질은 "현재 상태를 기준으로 어떤 단계를 다음에 실행할지 제어하는 것"이다.

## 정리

이번 노트에서 기억하면 좋은 문장은 아래 다섯 개다.

- Graph는 workflow 전체의 지도다
- `StateGraph`는 공유 상태를 중심으로 node를 연결한다
- node는 state를 읽고 partial update를 반환한다
- edge는 다음 실행 경로를 연결한다
- `START`와 `END`는 실행의 시작과 끝을 나타낸다

다음 단계에서는 조건 분기, router node, loop를 넣어서 상태 기반 orchestration다운 흐름으로 확장해보면 된다.